0. Libraries

In [23]:
# %% Cell 1 — Imports & Config

import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from tqdm.auto import tqdm  # progress bars (works in Jupyter + scripts)

# ---- Paths ----
INPUT_XLSX       = "combined_daily.xlsx"
FEATURE_CFG_JSON = "features_dictionary_full.json"

OUTDIR = Path("./feature_engineering_outputs")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ---- Core config ----
TOPN_TICKERS   = 100
RANDOM_SEED    = 42
LOOKBACK_YEARS = 10

RNG = np.random.default_rng(RANDOM_SEED)

WINDOWS = {
    "1m": 21,
    "3m": 63,
    "6m": 126,
    "12m": 252
}


1. Utility Functions

In [24]:
# %% Cell 2 — Utility Functions

def _to_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df.index, pd.DatetimeIndex):
        for c in df.columns:
            if str(c).lower() == "date":
                df = df.set_index(pd.to_datetime(df.pop(c)))
                break
    return df.sort_index()

def _safe_cols(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, ~df.columns.duplicated()]

def _log_ret(price, periods=1):
    return np.log(price / price.shift(periods))

def _rolling_sharpe(ret, win):
    mu = ret.rolling(win).mean()
    sig = ret.rolling(win).std(ddof=0)
    return (mu / sig) * np.sqrt(252)

def _rolling_sortino(ret, win):
    neg = ret.clip(upper=0.0)
    dn_vol = np.sqrt((neg**2).rolling(win).mean())
    mu = ret.rolling(win).mean()
    return (mu / dn_vol) * np.sqrt(252)

def _max_drawdown(ser, win):
    roll = ser.rolling(win)
    cummax = roll.max()
    dd = ser / cummax - 1.0
    return dd.rolling(win).min()

def _drawdown_series(price):
    peak = price.cummax()
    return price/peak - 1.0

def _zscore(ser, win):
    mu = ser.rolling(win).mean()
    sd = ser.rolling(win).std(ddof=0)
    return (ser - mu) / sd

def _autocorr(ret, win, lag=1):
    return ret.rolling(win).apply(lambda x: pd.Series(x).autocorr(lag=lag), raw=False)

def _rolling_var_cvar(ret, win, alpha=0.01):
    def _var(a):
        return np.nan if len(a)==0 else np.nanpercentile(a, 100*alpha)
    def _cvar(a):
        if len(a)==0:
            return np.nan
        v = np.nanpercentile(a, 100*alpha)
        tail = a[a<=v]
        return np.nan if len(tail)==0 else tail.mean()
    vr = ret.rolling(win).apply(_var, raw=True)
    cv = ret.rolling(win).apply(_cvar, raw=True)
    return vr, cv

def _minmax(ser, win):
    mn = ser.rolling(win).min()
    mx = ser.rolling(win).max()
    return (ser - mn) / (mx - mn)

def _cs_rank(df_row):
    return df_row.rank(axis=1, method="average", pct=True)

def _cs_zscore(df_row):
    mu = df_row.mean(axis=1)
    sd = df_row.std(axis=1, ddof=0)
    return (df_row.sub(mu, axis=0)).div(sd, axis=0)

def _best_market_proxy(prices, ff):
    if isinstance(ff, pd.DataFrame) and ("FF_MKT" in ff.columns):
        return ff["FF_MKT"]
    ret = prices.pct_change()
    return ret.mean(axis=1)

def _ols_xy(X, y):
    X_ = np.column_stack([np.ones(len(X)), X])
    beta, _, _, _ = np.linalg.lstsq(X_, y, rcond=None)
    yhat = X_.dot(beta)
    ssr = np.sum((yhat - y.mean())**2)
    sst = np.sum((y - y.mean())**2)
    r2 = np.nan if sst==0 else ssr/sst
    return beta[0], beta[1:], r2

def _rolling_ff_exposure(excess_ret: pd.Series, ff_df: pd.DataFrame, win: int):
    use_cols = [c for c in ["FF_MKT","FF_SMB","FF_HML","FF_RMW","FF_CMA"] if c in ff_df.columns]
    if not use_cols:
        # alpha + 5 betas + r2 placeholders
        alpha = pd.Series(np.nan, index=excess_ret.index)
        betas = [pd.Series(np.nan, index=excess_ret.index) for _ in range(5)]
        r2 = pd.Series(np.nan, index=excess_ret.index)
        return (alpha, *betas, r2)

    X = ff_df[use_cols]
    idx = excess_ret.index.intersection(X.index)
    ex = excess_ret.loc[idx]
    X = X.loc[idx]

    alpha = pd.Series(np.nan, index=idx)
    betas = {c: pd.Series(np.nan, index=idx) for c in use_cols}
    r2 = pd.Series(np.nan, index=idx)

    for t in range(win, len(idx)):
        sl = idx[t-win:t]
        y = ex.loc[sl].values
        M = X.loc[sl].values
        a, b, r = _ols_xy(M, y)
        alpha.iloc[t] = a
        r2.iloc[t] = r
        for i, c in enumerate(use_cols):
            betas[c].iloc[t] = b[i]
    return alpha, *(betas[c] for c in use_cols), r2


2. Load Data + Feature Config

In [25]:
# %% Cell 3 — Load Excel (fixed) + Load JSON

def load_combined(path: str):
    xls = pd.ExcelFile(path)
    sheets_lower = {s.lower(): s for s in xls.sheet_names}

    if {"prices", "volumes", "ff", "gw"}.issubset(sheets_lower.keys()):
        prices  = _to_datetime_index(_safe_cols(pd.read_excel(path, sheet_name=sheets_lower["prices"])))
        volumes = _to_datetime_index(_safe_cols(pd.read_excel(path, sheet_name=sheets_lower["volumes"])))
        ff      = _to_datetime_index(_safe_cols(pd.read_excel(path, sheet_name=sheets_lower["ff"])))
        gw      = _to_datetime_index(_safe_cols(pd.read_excel(path, sheet_name=sheets_lower["gw"])))
    else:
        df = _to_datetime_index(_safe_cols(pd.read_excel(path)))
        ff = df[[c for c in df.columns if str(c).startswith("FF_")]]
        gw = df[[c for c in df.columns if (str(c).lower() in {
            "dp","dy","ep","ntis","tbl","lty","ltr","tms","dfy","dfr","infl","reall","cay","vix",
            "ip_growth","cons_growth"
        }) or str(c).startswith("GW_")]]
        vol_cols = [c for c in df.columns if str(c).lower().startswith(("vol_","volume_"))]
        volumes = df[vol_cols].copy() if vol_cols else pd.DataFrame(index=df.index)
        prices  = df.drop(columns=ff.columns.union(gw.columns).union(volumes.columns), errors="ignore")

    prices  = prices.apply(pd.to_numeric, errors="coerce")
    volumes = volumes.apply(pd.to_numeric, errors="coerce")
    ff      = ff.apply(pd.to_numeric, errors="coerce")
    gw      = gw.apply(pd.to_numeric, errors="coerce")
    return prices, volumes, ff, gw

prices, volumes, ff, gw = load_combined(INPUT_XLSX)
print("Loaded data:")
print("  prices :", prices.shape)
print("  volumes:", volumes.shape)
print("  ff     :", ff.shape)
print("  gw     :", gw.shape)

with open(FEATURE_CFG_JSON, "r") as f:
    FEATURE_CFG = json.load(f)
print("Loaded feature config from", FEATURE_CFG_JSON)


Loaded data:
  prices : (6434, 503)
  volumes: (6434, 504)
  ff     : (6434, 10)
  gw     : (6434, 15)
Loaded feature config from features_dictionary_full.json


3. Feature Builder Functions

In [26]:
# %% Cell 4 — Filter last 10 years + select 100 tickers first

end_date = prices.index.max()
start_date = end_date - pd.DateOffset(years=LOOKBACK_YEARS)

prices_10y  = prices.loc[prices.index >= start_date].copy()
volumes_10y = volumes.loc[volumes.index >= start_date].copy()
ff_10y      = ff.loc[ff.index >= start_date].copy()
gw_10y      = gw.loc[gw.index >= start_date].copy()

# select tickers with enough data
valid_cols = []
for c in prices_10y.columns:
    s = prices_10y[c]
    if s.notna().sum() >= int(0.8 * len(prices_10y)):  # at least 80% coverage
        valid_cols.append(c)

if len(valid_cols) < TOPN_TICKERS:
    raise ValueError(f"Not enough tickers with sufficient history. Have {len(valid_cols)} need {TOPN_TICKERS}.")

selected_tickers = sorted(RNG.choice(valid_cols, size=TOPN_TICKERS, replace=False))
print("Selected 100 tickers (first 10):", selected_tickers[:10])
print("10y window:", prices_10y.index.min().date(), "->", prices_10y.index.max().date())

# filter all to those tickers
prices_10y = prices_10y[selected_tickers]

# try to filter volumes columns if they are keyed by ticker
if not volumes_10y.empty:
    if set(selected_tickers).issubset(set(volumes_10y.columns)):
        volumes_10y = volumes_10y[selected_tickers]
    else:
        # If your volume columns are like "VOLUME_AAPL" etc.
        cols = []
        for c in volumes_10y.columns:
            tick = str(c).split("__")[-1].split("_")[-1]
            if tick in selected_tickers:
                cols.append(c)
        volumes_10y = volumes_10y[cols] if cols else pd.DataFrame(index=prices_10y.index)

print("Filtered 10y subset:")
print("  prices :", prices_10y.shape)
print("  volumes:", volumes_10y.shape)
print("  ff     :", ff_10y.shape)
print("  gw     :", gw_10y.shape)


Selected 100 tickers (first 10): ['AAPL', 'AMP', 'AMT', 'AON', 'APA', 'APTV', 'ATO', 'AVGO', 'AXON', 'BMY']
10y window: 2015-06-30 -> 2025-06-30
Filtered 10y subset:
  prices : (2515, 100)
  volumes: (2515, 100)
  ff     : (2515, 10)
  gw     : (2515, 15)


4. Build Features & Targets on the Subset, Save

In [27]:
# %% Cell 5 — Feature builders (with progress bars)

def build_price_features(prices: pd.DataFrame) -> pd.DataFrame:
    ret_1d   = prices.pct_change(1).add_prefix("ret_1d__")
    ret_5d   = prices.pct_change(5).add_prefix("ret_5d__")
    ret_21d  = prices.pct_change(WINDOWS["1m"]).add_prefix("ret_21d__")
    log_ret  = _log_ret(prices, 1).add_prefix("log_ret_1d__")

    lags = {f"ret_lag_{m}m__": prices.pct_change(WINDOWS["1m"]*m) for m in [1,2,3,6,9,12]}

    mom = lambda m: prices / prices.shift(WINDOWS[m]) - 1.0
    momentum = {
        "momentum_1m__": mom("1m"),
        "momentum_3m__": mom("3m"),
        "momentum_6m__": mom("6m"),
        "momentum_12m__": mom("12m"),
    }

    r1 = prices.pct_change()
    vol_21  = r1.rolling(WINDOWS["1m"]).std(ddof=0).add_prefix("volatility_21d__")
    vol_63  = r1.rolling(WINDOWS["3m"]).std(ddof=0).add_prefix("volatility_63d__")
    vol_126 = r1.rolling(WINDOWS["6m"]).std(ddof=0).add_prefix("volatility_126d__")

    skew_63 = r1.rolling(WINDOWS["3m"]).skew().add_prefix("skewness_63d__")
    kurt_63 = r1.rolling(WINDOWS["3m"]).kurt().add_prefix("kurtosis_63d__")

    sma_21  = prices.rolling(WINDOWS["1m"]).mean().add_prefix("sma_21__")
    sma_63  = prices.rolling(WINDOWS["3m"]).mean().add_prefix("sma_63__")
    sma_126 = prices.rolling(WINDOWS["6m"]).mean().add_prefix("sma_126__")

    ma_ratio_21_63 = (sma_21.values / (sma_63.replace(0,np.nan).values)).astype(float)
    ma_ratio_21_63 = pd.DataFrame(ma_ratio_21_63, index=prices.index,
                                  columns=[f"ma_ratio_21_63__{c}" for c in prices.columns])
    ma_ratio_63_126 = (sma_63.values / (sma_126.replace(0,np.nan).values)).astype(float)
    ma_ratio_63_126 = pd.DataFrame(ma_ratio_63_126, index=prices.index,
                                   columns=[f"ma_ratio_63_126__{c}" for c in prices.columns])

    dd_63 = prices.apply(_drawdown_series).rolling(WINDOWS["3m"]).min().add_prefix("drawdown_63d__")
    z_price_63 = prices.apply(lambda s: _zscore(s, WINDOWS["3m"])).add_prefix("zscore_price_63d__")
    sharpe_63 = _rolling_sharpe(r1, WINDOWS["3m"]).add_prefix("rolling_sharpe_63d__")
    sortino_63 = _rolling_sortino(r1, WINDOWS["3m"]).add_prefix("rolling_sortino_63d__")
    ac_63 = r1.apply(lambda s: _autocorr(s, WINDOWS["3m"], lag=1)).add_prefix("autocorr_63d__")

    return pd.concat([
        ret_1d, ret_5d, ret_21d, log_ret,
        *lags.values(),
        *momentum.values(),
        vol_21, vol_63, vol_126,
        skew_63, kurt_63,
        sma_21, sma_63, sma_126,
        ma_ratio_21_63, ma_ratio_63_126,
        dd_63, z_price_63, sharpe_63, sortino_63, ac_63
    ], axis=1)


def build_volume_features(prices: pd.DataFrame, volumes: pd.DataFrame) -> pd.DataFrame:
    if volumes.empty:
        return pd.DataFrame(index=prices.index)

    v21  = volumes.rolling(WINDOWS["1m"]).mean().add_prefix("volume_21d_avg__")
    v63  = volumes.rolling(WINDOWS["3m"]).mean().add_prefix("volume_63d_avg__")
    turnover = (volumes / volumes.rolling(WINDOWS["1m"]).mean()).add_prefix("turnover__")
    vvol_21 = volumes.pct_change().rolling(WINDOWS["1m"]).std(ddof=0).add_prefix("volume_volatility_21d__")
    vvol_63 = volumes.pct_change().rolling(WINDOWS["3m"]).std(ddof=0).add_prefix("volume_volatility_63d__")

    r1 = prices.pct_change().fillna(0)
    pvt = (r1 * volumes).cumsum().add_prefix("pvt__")
    obv = (np.sign(prices.diff().fillna(0)) * volumes.fillna(0)).cumsum().add_prefix("obv__")

    mfv = (prices - prices.shift(1)) * volumes
    adl = mfv.cumsum().add_prefix("adl__")
    cmf = (mfv.rolling(WINDOWS["1m"]).sum() / (volumes.rolling(WINDOWS["1m"]).sum() + 1e-9)).add_prefix("cmf__")

    return pd.concat([v21, v63, turnover, vvol_21, vvol_63, pvt, obv, adl, cmf], axis=1)


def build_gw_features(gw: pd.DataFrame) -> pd.DataFrame:
    if gw.empty:
        return gw
    base = gw.copy()
    feats = [base.add_prefix("")]  # identity
    for k, w in [("lag1", WINDOWS["1m"]), ("lag3", WINDOWS["3m"]), ("lag6", WINDOWS["6m"])]:
        feats.append(base.shift(w).add_prefix(f"{k}_"))
    return pd.concat(feats, axis=1)


def build_ff_exposures(prices: pd.DataFrame, ff: pd.DataFrame) -> pd.DataFrame:
    if ff.empty or ("FF_RF" not in ff.columns):
        return pd.DataFrame(index=prices.index)

    ret = prices.pct_change()
    exret = ret.sub(ff["FF_RF"], axis=0)

    out = {}
    for c in tqdm(prices.columns, desc="FF exposures (rolling regressions)"):
        a6,*b6,r26    = _rolling_ff_exposure(exret[c], ff, WINDOWS["6m"])
        a12,*b12,r212 = _rolling_ff_exposure(exret[c], ff, WINDOWS["12m"])

        cols = [col for col in ["FF_MKT","FF_SMB","FF_HML","FF_RMW","FF_CMA"] if col in ff.columns]

        out[f"alpha_FF_6m__{c}"] = a6
        for k, b in zip(cols, b6):
            out[f"beta_{k.split('_')[1]}_6m__{c}"] = b
        out[f"r2_FF_6m__{c}"] = r26

        out[f"alpha_FF_12m__{c}"] = a12
        for k, b in zip(cols, b12):
            out[f"beta_{k.split('_')[1]}_12m__{c}"] = b
        out[f"r2_FF_12m__{c}"] = r212

    return pd.DataFrame(out).sort_index()


def build_risk_features(prices: pd.DataFrame) -> pd.DataFrame:
    ret = prices.pct_change()

    var1, cvar1 = _rolling_var_cvar(ret, WINDOWS["3m"], alpha=0.01)
    var5, cvar5 = _rolling_var_cvar(ret, WINDOWS["3m"], alpha=0.05)

    var1  = var1.add_prefix("rolling_var_63d_1p__")
    cvar1 = cvar1.add_prefix("rolling_cvar_63d_1p__")
    var5  = var5.add_prefix("rolling_var_63d_5p__")
    cvar5 = cvar5.add_prefix("rolling_cvar_63d_5p__")

    semi_var = ret.clip(upper=0).pow(2).rolling(WINDOWS["3m"]).mean().add_prefix("semi_variance_63d__")
    mdd_126  = prices.apply(lambda s: _max_drawdown(s, WINDOWS["6m"])).add_prefix("max_drawdown_126d__")
    down_vol = ret.clip(upper=0).rolling(WINDOWS["3m"]).std(ddof=0).add_prefix("downside_volatility_63d__")

    return pd.concat([var1, var5, cvar1, cvar5, semi_var, mdd_126, down_vol], axis=1)


def build_cross_sectional_features(prices: pd.DataFrame) -> pd.DataFrame:
    ret_21 = prices.pct_change(WINDOWS["1m"])
    vol_63 = prices.pct_change().rolling(WINDOWS["3m"]).std(ddof=0)

    cs_rank_mom = _cs_rank(ret_21).add_prefix("cross_sectional_rank_momentum__")
    cs_rank_vol = _cs_rank(vol_63).add_prefix("cross_sectional_rank_volatility__")
    return pd.concat([cs_rank_mom, cs_rank_vol], axis=1)


def build_derived_stats(prices: pd.DataFrame, ff: pd.DataFrame) -> pd.DataFrame:
    ret = prices.pct_change()
    mkt = _best_market_proxy(prices, ff)

    corr63 = pd.DataFrame({f"rolling_corr_market_63d__{c}": ret[c].rolling(WINDOWS["3m"]).corr(mkt) for c in prices.columns})

    pc1 = pd.DataFrame(index=prices.index, columns=[f"pca_component_1__{c}" for c in prices.columns], dtype=float)
    win = WINDOWS["3m"]

    for t in tqdm(range(win, len(prices)), desc="Rolling PCA"):
        sl = slice(prices.index[t-win], prices.index[t-1])
        R = ret.loc[sl, :].copy()
        if R.isna().all().all():
            continue
        R = (R - R.mean())/R.std(ddof=0)
        R = R.replace([np.inf, -np.inf], np.nan).dropna(axis=1, thresh=int(0.8*len(R))).fillna(0.0)
        if R.shape[1] < 5:
            continue
        pca = PCA(n_components=1)
        comp = pca.fit(R).components_.T
        cols = list(R.columns)
        idx = prices.index[t]
        for i, col in enumerate(cols):
            pc1.at[idx, f"pca_component_1__{col}"] = comp[i,0]

    return pd.concat([corr63, pc1], axis=1)


def build_transformations(df_features: pd.DataFrame, base_cols_subset=None) -> pd.DataFrame:
    X = df_features.copy()
    if base_cols_subset is None:
        base_cols_subset = [c for c in X.columns if c.startswith(("momentum_","volatility_","semi_variance_","downside_volatility_"))]
        base_cols_subset = base_cols_subset[: min(80, len(base_cols_subset))]

    parts = []
    for c in tqdm(base_cols_subset, desc="Transformations"):
        s = X[c]
        parts.append(((s - s.mean())/s.std(ddof=0)).rename(f"standardized__{c}"))
        parts.append(_minmax(s, WINDOWS["6m"]).rename(f"minmax__{c}"))
        parts.append(np.log1p(s.clip(lower=0)).rename(f"log__{c}"))
    return pd.concat(parts, axis=1) if parts else pd.DataFrame(index=X.index)


def build_targets(prices: pd.DataFrame, ff: pd.DataFrame, topn=TOPN_TICKERS) -> pd.DataFrame:
    fwd = prices.shift(-WINDOWS["1m"])
    ret_next1m = (fwd / prices - 1.0).add_prefix("target_return_next1m__")

    if not ff.empty and ("FF_RF" in ff.columns):
        target_ex = ret_next1m.sub(ff["FF_RF"].shift(-WINDOWS["1m"]), axis=0)\
                              .rename(columns=lambda c: c.replace("target_return_next1m__", "target_excess_return_next1m__"))
    else:
        target_ex = pd.DataFrame(index=prices.index)

    return pd.concat([ret_next1m, target_ex], axis=1)


5. Build features ONLY for 10y + 100 tickers

In [28]:
# %% Cell 6 — Build features for 10y + 100 tickers (fast)

F_price = build_price_features(prices_10y)
F_vol   = build_volume_features(prices_10y, volumes_10y)
F_ff    = build_ff_exposures(prices_10y, ff_10y)
F_gw    = build_gw_features(gw_10y)
F_risk  = build_risk_features(prices_10y)
F_cs    = build_cross_sectional_features(prices_10y)
F_der   = build_derived_stats(prices_10y, ff_10y)

F_all_pre = pd.concat([F_price, F_vol, F_ff, F_risk, F_cs, F_der], axis=1)
F_tr = build_transformations(F_all_pre)

Features = pd.concat([F_price, F_vol, F_ff, F_gw, F_risk, F_cs, F_der, F_tr], axis=1)
Features = Features.loc[:, ~Features.columns.duplicated()].sort_index()

T = build_targets(prices_10y, ff_10y, topn=TOPN_TICKERS)

print("✅ Built Features:", Features.shape)
print("✅ Built Targets :", T.shape)

Features.to_parquet(OUTDIR / "features_10y_100tickers_wide.parquet")
T.to_parquet(OUTDIR / "targets_10y_100tickers_wide.parquet")
print("Saved parquet files.")


FF exposures (rolling regressions):   0%|          | 0/100 [00:00<?, ?it/s]

Rolling PCA:   0%|          | 0/2452 [00:00<?, ?it/s]

Transformations:   0%|          | 0/80 [00:00<?, ?it/s]

✅ Built Features: (2515, 5100)
✅ Built Targets : (2515, 200)
Saved parquet files.


6. Build panel with progress bar

In [29]:
# %% Cell 7 — Build panel (Date,Ticker) robustly (fix duplicate columns)

def build_panel_from_wide(Features: pd.DataFrame,
                          T: pd.DataFrame,
                          target_prefix="target_return_next1m__"):

    # ---- Targets wide -> long ----
    y_cols = [c for c in T.columns if c.startswith(target_prefix)]
    if not y_cols:
        raise ValueError(f"No target columns with prefix '{target_prefix}' found in T.")

    Y_wide = T[y_cols].copy()
    # columns like target_return_next1m__AAPL -> AAPL
    Y_wide.columns = [c.split("__", 1)[1] if "__" in c else c for c in Y_wide.columns]
    Y_long = (Y_wide.stack()
                    .rename("y")
                    .reset_index())
    Y_long.columns = ["Date", "Ticker", "y"]

    # ---- Build mapping from wide feature columns -> (feature_name, ticker) ----
    records = []
    for col in Features.columns:
        if "__" not in col:
            continue
        feat, tick = col.split("__", 1)
        records.append((tick, feat, col))

    rec_df = pd.DataFrame(records, columns=["Ticker", "Feature", "ColName"])

    if rec_df.empty:
        raise ValueError("No ticker-level feature columns found in Features (expected 'feature__TICKER').")

    # ---- Build X_long per ticker (deduplicate columns per ticker) ----
    frames = []
    for tick, sub in tqdm(rec_df.groupby("Ticker"), desc="Building panel per ticker"):
        cols = sub["ColName"].tolist()
        feat_names = sub["Feature"].tolist()

        tmp = Features[cols].copy()
        tmp.columns = feat_names  # rename to base feature names

        # ✅ critical fix: drop duplicate columns within this ticker
        # (keeps first occurrence)
        tmp = tmp.loc[:, ~pd.Index(tmp.columns).duplicated(keep="first")]

        tmp["Ticker"] = tick
        tmp = tmp.reset_index()
        tmp = tmp.rename(columns={tmp.columns[0]: "Date"})  # index -> Date
        frames.append(tmp)

    # concat is now safe because every tmp has unique columns
    X_long = pd.concat(frames, axis=0, ignore_index=True, sort=False)

    # merge (Date,Ticker)
    panel = pd.merge(Y_long, X_long, on=["Date", "Ticker"], how="inner")
    panel = panel.set_index(["Date", "Ticker"]).sort_index()

    # keep only numeric feature columns
    for c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

    return panel


panel = build_panel_from_wide(Features, T, target_prefix="target_return_next1m__")
print("✅ Panel shape:", panel.shape)
print("✅ Panel columns:", len(panel.columns), "(including y)")


Building panel per ticker:   0%|          | 0/180 [00:00<?, ?it/s]

✅ Panel shape: (249400, 51)
✅ Panel columns: 51 (including y)


In [30]:
# ---- CLEAN PANEL: replace inf with NaN and optionally cap extremes ----
panel = panel.replace([np.inf, -np.inf], np.nan)

# Optional (recommended): winsorize/cap extreme feature values to reduce numeric explosions
# This caps features at the 0.1% / 99.9% quantiles (cross-sectionally over the panel)
feat_cols = [c for c in panel.columns if c != "y"]
q_low = panel[feat_cols].quantile(0.001)
q_hi  = panel[feat_cols].quantile(0.999)
panel[feat_cols] = panel[feat_cols].clip(lower=q_low, upper=q_hi, axis=1)

print("✅ Panel cleaned: inf -> NaN, winsorized extremes")


✅ Panel cleaned: inf -> NaN, winsorized extremes


7. tests + Top 20 (with progress bar)

In [31]:
# %% Cell 8 — 8 Tests + Top 20 Features (tqdm)

def interpretability_score_from_name(name: str) -> float:
    if name.startswith(("momentum_", "ret_", "log_ret_")): return 1.0
    if name.startswith(("volatility_", "downside_volatility_", "semi_variance_")): return 0.9
    if name.startswith(("rolling_var_", "rolling_cvar_")): return 0.9
    if name.startswith(("cross_sectional_", "relative_")): return 0.8
    if name.startswith(("max_drawdown_", "drawdown_")): return 0.8
    if name.startswith(("pca_component_", "copula_", "systemic_", "regime_")): return 0.4
    if name.startswith(("sma_", "ma_ratio_")): return 0.7
    if name.startswith(("rolling_corr_",)): return 0.6
    return 0.5

def compute_feature_quality(panel: pd.DataFrame) -> pd.DataFrame:
    y = panel["y"]
    X = panel.drop(columns=["y"])
    n_total = len(panel)

    dates = panel.index.get_level_values("Date")
    month_key = dates.to_period("M")

    corr_mat = X.corr(method="pearson")
    np.fill_diagonal(corr_mat.values, 0.0)
    max_abs_corr = corr_mat.abs().max(axis=1)

    unique_dates = np.sort(dates.unique())
    split_idx = int(0.7 * len(unique_dates))
    train_cutoff = unique_dates[split_idx]
    mask_train = dates <= train_cutoff
    mask_test  = dates > train_cutoff

    rows = []
    for feat in tqdm(X.columns, desc="Running 8 feature tests"):
        s = X[feat]
        mask_valid = s.notna() & y.notna()
        n_valid = mask_valid.sum()
        missing_ratio = 1.0 - n_valid / n_total

        if n_valid < 200:
            rows.append({"feature": feat, "missing_ratio": missing_ratio, "std": np.nan,
                         "spearman_corr": np.nan, "pearson_corr": np.nan, "mutual_info": np.nan,
                         "ic_mean": np.nan, "ic_ir": np.nan, "cs_r2_mean": np.nan, "oos_r2": np.nan,
                         "max_abs_corr": float(max_abs_corr.get(feat, np.nan)),
                         "interpretability_score": interpretability_score_from_name(feat),
                         "score": -np.inf})
            continue

        s_valid = s[mask_valid]
        y_valid = y[mask_valid]

        std_val = float(s_valid.std(ddof=0))
        spearman = float(s_valid.corr(y_valid, method="spearman"))
        pearson  = float(s_valid.corr(y_valid, method="pearson"))

        mi_val = np.nan
        try:
            mi_val = float(mutual_info_regression(s_valid.values.reshape(-1,1), y_valid.values)[0])
        except Exception:
            pass

        df_m = pd.DataFrame({"y": y_valid, "feat": s_valid, "month": month_key[mask_valid]}).dropna()

        def _ic(g):
            if g["feat"].nunique() < 2 or g["y"].nunique() < 2:
                return np.nan
            return g["feat"].corr(g["y"], method="spearman")

        ic_by_month = df_m.groupby("month").apply(_ic).dropna()
        if len(ic_by_month) > 1:
            ic_mean = float(ic_by_month.mean())
            ic_std  = float(ic_by_month.std(ddof=0))
            ic_ir   = ic_mean / ic_std if ic_std > 0 else np.nan
        else:
            ic_mean, ic_ir = np.nan, np.nan

        cs_r2_list = []
        for _, g in df_m.groupby("month"):
            if g["feat"].nunique() < 2 or g["y"].nunique() < 2:
                continue
            c = g["feat"].corr(g["y"], method="pearson")
            if not pd.isna(c):
                cs_r2_list.append(c**2)
        cs_r2_mean = float(np.mean(cs_r2_list)) if cs_r2_list else np.nan

        # --- OOS R2 (robust to inf/NaN) ---
        train_mask = mask_valid & mask_train
        test_mask  = mask_valid & mask_test

        s_train = s[train_mask].replace([np.inf, -np.inf], np.nan)
        y_train = y[train_mask].replace([np.inf, -np.inf], np.nan)
        s_test  = s[test_mask].replace([np.inf, -np.inf], np.nan)
        y_test  = y[test_mask].replace([np.inf, -np.inf], np.nan)

        # drop NaNs after cleaning
        mtr = s_train.notna() & y_train.notna()
        mts = s_test.notna() & y_test.notna()

        if mtr.sum() > 50 and mts.sum() > 50:
            Xtr = s_train[mtr].values.reshape(-1, 1)
            Ytr = y_train[mtr].values
            Xts = s_test[mts].values.reshape(-1, 1)
            Yts = y_test[mts].values

            # extra safety: if feature is constant in train, skip
            if np.nanstd(Xtr) < 1e-12:
                oos_r2 = np.nan
            else:
                lr = LinearRegression().fit(Xtr, Ytr)
                y_pred = lr.predict(Xts)
                oos_r2 = float(r2_score(Yts, y_pred))
        else:
            oos_r2 = np.nan


        mac = float(max_abs_corr.get(feat, np.nan))
        interp = interpretability_score_from_name(feat)

        score_pred = abs(spearman) + (mi_val if not np.isnan(mi_val) else 0.0)
        score_stab = ic_ir if not np.isnan(ic_ir) else 0.0
        score_cs   = cs_r2_mean if not np.isnan(cs_r2_mean) else 0.0
        score_oos  = oos_r2 if not np.isnan(oos_r2) else 0.0
        score_qual = (1.0 - missing_ratio)
        score_red  = (1.0 - mac) if not np.isnan(mac) else 0.0

        final_score = (
            0.30 * score_pred +
            0.20 * score_stab +
            0.15 * score_cs +
            0.15 * score_oos +
            0.10 * score_qual +
            0.08 * score_red +
            0.02 * interp
        )

        rows.append({
            "feature": feat, "missing_ratio": missing_ratio, "std": std_val,
            "spearman_corr": spearman, "pearson_corr": pearson,
            "mutual_info": mi_val, "ic_mean": ic_mean, "ic_ir": ic_ir,
            "cs_r2_mean": cs_r2_mean, "oos_r2": oos_r2,
            "max_abs_corr": mac, "interpretability_score": interp,
            "score": final_score
        })

    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)

feature_report = compute_feature_quality(panel)

TOP_K = 30
top20 = feature_report.head(TOP_K)

display(top20[["feature","score","missing_ratio","std","spearman_corr","ic_mean","ic_ir","cs_r2_mean","oos_r2"]])

OUT_PATH = OUTDIR / "feature_quality_report_10y_100tickers.csv"
feature_report.to_csv(OUT_PATH, index=False)
print("✅ Saved report:", OUT_PATH.resolve())

print("\nTop 20 features:")
for f in top20["feature"].tolist():
    print(" -", f)


Running 8 feature tests:   0%|          | 0/50 [00:00<?, ?it/s]

,feature,score,missing_ratio,std,spearman_corr,ic_mean,ic_ir,cs_r2_mean,oos_r2
0,semi_variance_63d,0.259408,0.025261,3.344023e-04,0.087627,0.061172,0.301484,0.044402,0.009222
1,volatility_21d,0.252541,0.008420,1.047430e-02,0.080569,0.048603,0.255337,0.037541,0.011550
2,downside_volatility_63d,0.252166,0.025261,6.122780e-03,0.084726,0.059358,0.301022,0.044153,0.005358
3,kurtosis_63d,0.251507,0.025261,5.377745e+00,0.018475,0.023863,0.283028,0.008489,-0.000465
4,volatility_126d,0.247262,0.050521,8.937861e-03,0.085780,0.050824,0.242729,0.052299,-0.000436
5,volatility_63d,0.246172,0.025261,9.569344e-03,0.082860,0.051045,0.247542,0.049534,0.006077
6,turnover,0.238809,0.008019,4.510599e-01,0.000156,0.015417,0.178744,0.005153,-0.000196
7,cross_sectional_rank_volatility,0.235068,0.025261,2.886596e-01,0.035839,0.043022,0.216378,0.042936,0.002845
8,volume_volatility_21d,0.228226,0.008589,2.560998e-01,0.016853,0.024042,0.280096,0.006164,0.000085
9,volume_volatility_63d,0.220336,0.025766,2.420585e-01,0.006298,0.020721,0.253903,0.008914,-0.000534


✅ Saved report: D:\Thesis\Old\Codes\New\feature_engineering_outputs\feature_quality_report_10y_100tickers.csv

Top 20 features:
 - semi_variance_63d
 - volatility_21d
 - downside_volatility_63d
 - kurtosis_63d
 - volatility_126d
 - volatility_63d
 - turnover
 - cross_sectional_rank_volatility
 - volume_volatility_21d
 - volume_volatility_63d
 - rolling_corr_market_63d
 - r2_FF_12m
 - r2_FF_6m
 - beta_SMB_6m
 - beta_SMB_12m
 - skewness_63d
 - pca_component_1
 - autocorr_63d
 - ma_ratio_63_126
 - drawdown_63d
 - volume_21d_avg
 - rolling_var_63d_5p
 - alpha_FF_12m
 - max_drawdown_126d
 - volume_63d_avg
 - sma_63
 - sma_21
 - sma_126
 - ma_ratio_21_63
 - alpha_FF_6m


ML and DL tests

In [13]:
# %% Cell 1 — Define ML and DL feature sets

# --- ML feature set (Top 10, low redundancy) ---
FEATURES_ML = [
    "semi_variance_63d",
    "volatility_21d",
    "downside_volatility_63d",
    "kurtosis_63d",
    "cross_sectional_rank_volatility",
    "rolling_corr_market_63d",
    "r2_FF_12m",
    "turnover",
    "volume_volatility_21d",
    "pca_component_1",
]

# --- DL feature set (expanded, richer representation) ---
FEATURES_DL = FEATURES_ML + [
    "volatility_63d",
    "volatility_126d",
    "beta_SMB_6m",
    "beta_SMB_12m",
    "skewness_63d",
    "ma_ratio_21_63",
    "ma_ratio_63_126",
    "autocorr_63d",
]

print("ML features:", len(FEATURES_ML))
print("DL features:", len(FEATURES_DL))


ML features: 10
DL features: 18


In [14]:
# %% Cell 2 — Build and save ML/DL datasets

from pathlib import Path

DATASET_DIR = OUTDIR / "model_datasets"
DATASET_DIR.mkdir(exist_ok=True)

def build_dataset(panel, feature_list, name):
    cols = ["y"] + feature_list
    df = panel[cols].copy()

    # final safety cleaning
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["y"])

    path = DATASET_DIR / name
    df.to_parquet(path)

    print(f"✅ Saved {name}")
    print("   Shape:", df.shape)
    return df

df_ml = build_dataset(
    panel,
    FEATURES_ML,
    "model_dataset_ml_top10.parquet"
)

df_dl = build_dataset(
    panel,
    FEATURES_DL,
    "model_dataset_dl_top18.parquet"
)


✅ Saved model_dataset_ml_top10.parquet
   Shape: (249400, 11)
✅ Saved model_dataset_dl_top18.parquet
   Shape: (249400, 19)


In [17]:
# %% Cell 3 — Chronological split

def time_split(df, split_ratio=0.7):
    df = df.reset_index().sort_values(["Date", "Ticker"])
    dates = np.sort(df["Date"].unique())
    cutoff = dates[int(len(dates) * split_ratio)]

    train = df[df["Date"] <= cutoff]
    test  = df[df["Date"] > cutoff]

    print("Cutoff date:", pd.to_datetime(cutoff).date())
    print("Train:", train.shape, " Test:", test.shape)
    return train, test

train_ml, test_ml = time_split(df_ml)


Cutoff date: 2022-06-03
Train: (174600, 13)  Test: (74800, 13)


ML

In [18]:
# %% Cell 4 — Prepare ML matrices

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

def prepare_xy(train, test, features):
    Xtr = train[features].values
    ytr = train["y"].values
    Xts = test[features].values
    yts = test["y"].values

    imp = SimpleImputer(strategy="median")
    Xtr = imp.fit_transform(Xtr)
    Xts = imp.transform(Xts)

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr)
    Xts = scaler.transform(Xts)

    return Xtr, ytr, Xts, yts

Xtr, ytr, Xts, yts = prepare_xy(train_ml, test_ml, FEATURES_ML)


In [49]:
# %% Cell 5 — ML training & evaluation

from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr

models = {
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1e-4, l1_ratio=0.5, max_iter=5000),
    "RandomForest": RandomForestRegressor(
        n_estimators=300, max_depth=None, random_state=42, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

results = []

for name, model in models.items():
    model.fit(Xtr, ytr)
    preds = model.predict(Xts)

    r2 = r2_score(yts, preds)
    mse = mean_squared_error(yts, preds)
    ic  = spearmanr(preds, yts, nan_policy="omit").correlation

    results.append({
        "model": name,
        "OOS_R2": r2,
        "MSE": mse,
        "IC": ic,
    })

ml_results = pd.DataFrame(results).sort_values("IC", ascending=False)
display(ml_results)


,model,OOS_R2,MSE,IC
1,ElasticNet,0.011285,0.008109,0.078363
0,Ridge,0.011180,0.008110,0.078298
3,GradientBoosting,0.002553,0.008180,0.049968
2,RandomForest,-0.080188,0.008859,0.026292


DL

In [19]:
# %% Cell 6 — Prepare DL tensors (no training yet)

train_dl, test_dl = time_split(df_dl)

Xtr_dl, ytr_dl, Xts_dl, yts_dl = prepare_xy(
    train_dl, test_dl, FEATURES_DL
)

print("DL train:", Xtr_dl.shape, " DL test:", Xts_dl.shape)


Cutoff date: 2022-06-03
Train: (174600, 21)  Test: (74800, 21)
DL train: (174600, 18)  DL test: (74800, 18)


In [53]:
# %% Cell 7 — Train a DL baseline (MLP) with early stopping

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# tensors
Xtr_t = torch.tensor(Xtr_dl, dtype=torch.float32)
ytr_t = torch.tensor(ytr_dl, dtype=torch.float32).view(-1, 1)
Xts_t = torch.tensor(Xts_dl, dtype=torch.float32)
yts_t = torch.tensor(yts_dl, dtype=torch.float32).view(-1, 1)

train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=4096, shuffle=True)

class MLP(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x)

model = MLP(Xtr_dl.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

# Early stopping settings
EPOCHS = 30
PATIENCE = 5
best_mse = np.inf
best_state = None
pat = 0

def eval_model(model):
    model.eval()
    with torch.no_grad():
        pred = model(Xts_t.to(device)).cpu().numpy().ravel()
    y_true = yts_dl
    mse = mean_squared_error(y_true, pred)
    r2  = r2_score(y_true, pred)
    ic  = spearmanr(pred, y_true, nan_policy="omit").correlation
    return mse, r2, ic, pred

history = []
for ep in range(1, EPOCHS + 1):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = loss_fn(out, yb)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item() * len(xb)

    train_mse = total / len(Xtr_t)
    val_mse, val_r2, val_ic, _ = eval_model(model)

    history.append({"epoch": ep, "train_mse": train_mse, "test_mse": val_mse, "test_r2": val_r2, "test_ic": val_ic})
    print(f"Epoch {ep:02d} | train_MSE={train_mse:.6f} | test_MSE={val_mse:.6f} | test_R2={val_r2:.4f} | test_IC={val_ic:.4f}")

    if val_mse < best_mse - 1e-6:
        best_mse = val_mse
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        pat = 0
    else:
        pat += 1
        if pat >= PATIENCE:
            print("⏹️ Early stopping triggered.")
            break

# Load best state
if best_state is not None:
    model.load_state_dict(best_state)

best_mse, best_r2, best_ic, best_pred = eval_model(model)

print("\n✅ Best DL MLP results")
print("Test MSE:", best_mse)
print("Test R2 :", best_r2)
print("Test IC :", best_ic)

dl_history = pd.DataFrame(history)
display(dl_history.tail(10))


Device: cpu
Epoch 01 | train_MSE=0.010729 | test_MSE=0.008372 | test_R2=-0.0208 | test_IC=0.0480
Epoch 02 | train_MSE=0.007940 | test_MSE=0.008354 | test_R2=-0.0186 | test_IC=0.0361
Epoch 03 | train_MSE=0.007586 | test_MSE=0.008385 | test_R2=-0.0224 | test_IC=0.0306
Epoch 04 | train_MSE=0.007403 | test_MSE=0.008356 | test_R2=-0.0189 | test_IC=0.0302
Epoch 05 | train_MSE=0.007237 | test_MSE=0.008412 | test_R2=-0.0257 | test_IC=0.0329
Epoch 06 | train_MSE=0.007142 | test_MSE=0.008444 | test_R2=-0.0296 | test_IC=0.0371
Epoch 07 | train_MSE=0.007028 | test_MSE=0.008494 | test_R2=-0.0356 | test_IC=0.0353
⏹️ Early stopping triggered.

✅ Best DL MLP results
Test MSE: 0.008353655810982921
Test R2 : -0.018572254139306077
Test IC : 0.03609820476893775


,epoch,train_mse,test_mse,test_r2,test_ic
0,1,0.010729,0.008372,-0.020840,0.048017
1,2,0.007940,0.008354,-0.018572,0.036098
2,3,0.007586,0.008385,-0.022397,0.030558
3,4,0.007403,0.008356,-0.018903,0.030210
4,5,0.007237,0.008412,-0.025712,0.032911
5,6,0.007142,0.008444,-0.029610,0.037096
6,7,0.007028,0.008494,-0.035631,0.035344


In [54]:
# %% Cell 8 — IC by date (cross-sectional) + IC-IR

# rebuild test dataframe with predictions
test_dl_pred = test_dl.copy()
test_dl_pred["pred"] = best_pred

# cross-sectional IC per day (Spearman across tickers)
ic_by_date = (
    test_dl_pred.groupby("Date")
               .apply(lambda g: g["pred"].corr(g["y"], method="spearman"))
               .dropna()
)

ic_mean = float(ic_by_date.mean())
ic_std  = float(ic_by_date.std(ddof=0))
ic_ir   = float(ic_mean / (ic_std + 1e-12))

print("✅ Test IC mean:", ic_mean)
print("✅ Test IC IR  :", ic_ir)
print("N dates used  :", len(ic_by_date))

# show distribution summary
display(ic_by_date.describe())


✅ Test IC mean: 0.02087107311129066
✅ Test IC IR  : 0.13481695953413864
N dates used  : 748


count    748.000000
mean       0.020871
std        0.154914
min       -0.434683
25%       -0.083474
50%        0.006061
75%        0.103879
max        0.583222
dtype: float64

In [55]:
# %% Cell 9 — Compare ML vs DL (headline metrics)

summary = pd.DataFrame([
    {"model": "DL_MLP", "OOS_R2": best_r2, "MSE": best_mse, "IC": best_ic}
])

try:
    summary = pd.concat([ml_results.rename(columns={"OOS_R2": "OOS_R2"}), summary], ignore_index=True)
except NameError:
    pass

display(summary.sort_values("IC", ascending=False))


,model,OOS_R2,MSE,IC
0,ElasticNet,0.011285,0.008109,0.078363
1,Ridge,0.011180,0.008110,0.078298
2,GradientBoosting,0.002553,0.008180,0.049968
4,DL_MLP,-0.018572,0.008354,0.036098
3,RandomForest,-0.080188,0.008859,0.026292


In [56]:
# %% Cell 10 — Save test predictions + final dataset snapshot

pred_path = DATASET_DIR / "dl_predictions_test.parquet"
test_dl_pred[["Date", "Ticker", "y", "pred"]].to_parquet(pred_path)

print("✅ Saved DL predictions:", pred_path.resolve())


✅ Saved DL predictions: D:\Thesis\Old\Codes\New\feature_engineering_outputs\model_datasets\dl_predictions_test.parquet


In [57]:
# %% Cell 7 — DL baselines: MLP (tabular) + 1D-CNN (sequence over time)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

LOOKBACK = 60          # <- 60 trading days is a good default
BATCH_SIZE = 4096
EPOCHS = 30
PATIENCE = 5
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------
def eval_preds(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    r2  = r2_score(y_true, y_pred)
    ic  = spearmanr(y_pred, y_true, nan_policy="omit").correlation
    return mse, r2, ic

def early_stop_train(model, train_loader, X_test_t, y_test_np, lr=1e-3, weight_decay=1e-4):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_mse = np.inf
    best_state = None
    pat = 0
    history = []

    for ep in range(1, EPOCHS + 1):
        model.train()
        total = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item() * len(xb)

        train_mse = total / len(train_loader.dataset)

        model.eval()
        with torch.no_grad():
            pred_test = model(X_test_t.to(device)).cpu().numpy().ravel()

        test_mse, test_r2, test_ic = eval_preds(y_test_np, pred_test)

        history.append({"epoch": ep, "train_mse": train_mse, "test_mse": test_mse, "test_r2": test_r2, "test_ic": test_ic})
        print(f"Epoch {ep:02d} | train_MSE={train_mse:.6f} | test_MSE={test_mse:.6f} | test_R2={test_r2:.4f} | test_IC={test_ic:.4f}")

        if test_mse < best_mse - 1e-6:
            best_mse = test_mse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            pat = 0
        else:
            pat += 1
            if pat >= PATIENCE:
                print("⏹️ Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        best_pred = model(X_test_t.to(device)).cpu().numpy().ravel()

    best_mse, best_r2, best_ic = eval_preds(y_test_np, best_pred)
    return best_mse, best_r2, best_ic, best_pred, pd.DataFrame(history)

# ---------------------------------------------------------------------
# 1) TABULAR MLP (uses Xtr_dl/Xts_dl already prepared from Cell 6)
# ---------------------------------------------------------------------
Xtr_tab = torch.tensor(Xtr_dl, dtype=torch.float32)
ytr_tab = torch.tensor(ytr_dl, dtype=torch.float32).view(-1, 1)
Xts_tab = torch.tensor(Xts_dl, dtype=torch.float32)
yts_np  = np.asarray(yts_dl).ravel()

tab_loader = DataLoader(TensorDataset(Xtr_tab, ytr_tab), batch_size=BATCH_SIZE, shuffle=True)

class MLP(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x)

print("\n====================")
print("Training: TABULAR MLP")
print("====================")
mlp = MLP(Xtr_dl.shape[1]).to(device)
mlp_mse, mlp_r2, mlp_ic, mlp_pred, mlp_hist = early_stop_train(
    mlp, tab_loader, Xts_tab, yts_np, lr=1e-3, weight_decay=1e-4
)

print("\n✅ Best TABULAR MLP")
print("Test MSE:", mlp_mse)
print("Test R2 :", mlp_r2)
print("Test IC :", mlp_ic)


# ---------------------------------------------------------------------
# 2) SEQUENCE BUILDER for 1D-CNN (convolve over TIME)
# ---------------------------------------------------------------------
def build_sequences_from_df(df, features, lookback):
    """
    df must have columns: Date, Ticker, y, <features...>
    returns:
      X_seq: (N, lookback, F)
      y_seq: (N,)
      meta : DataFrame with Date,Ticker for each sample (aligned to y)
    """
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    X_list, y_list, meta_rows = [], [], []

    for tick, g in df.groupby("Ticker", sort=False):
        g = g.sort_values("Date")
        X = g[features].values
        y = g["y"].values
        dates = g["Date"].values

        # rolling windows
        for t in range(lookback - 1, len(g)):
            x_win = X[t - lookback + 1 : t + 1]
            y_t   = y[t]
            if np.any(np.isnan(x_win)) or np.isnan(y_t):
                continue
            X_list.append(x_win)
            y_list.append(y_t)
            meta_rows.append((dates[t], tick))

    X_seq = np.asarray(X_list, dtype=np.float32)
    y_seq = np.asarray(y_list, dtype=np.float32)
    meta  = pd.DataFrame(meta_rows, columns=["Date", "Ticker"])
    return X_seq, y_seq, meta


# ---------------------------------------------------------------------
# 3) Prepare DL train/test dataframes (need Date/Ticker/y/features)
#    IMPORTANT: fit imputer+scaler on TRAIN ONLY, then transform both.
# ---------------------------------------------------------------------
train_df = train_dl.copy()
test_df  = test_dl.copy()

# Impute + scale (fit on train only)
imp = SimpleImputer(strategy="median")
scaler = StandardScaler()

Xtr_raw = imp.fit_transform(train_df[FEATURES_DL].values)
Xts_raw = imp.transform(test_df[FEATURES_DL].values)

Xtr_scaled = scaler.fit_transform(Xtr_raw)
Xts_scaled = scaler.transform(Xts_raw)

# put back scaled features into frames
train_df_scaled = train_df[["Date","Ticker","y"]].copy()
test_df_scaled  = test_df[["Date","Ticker","y"]].copy()

for i, f in enumerate(FEATURES_DL):
    train_df_scaled[f] = Xtr_scaled[:, i]
    test_df_scaled[f]  = Xts_scaled[:, i]

# Build sequences separately for train and test (chronological already enforced by your split)
Xtr_seq, ytr_seq, meta_tr = build_sequences_from_df(train_df_scaled, FEATURES_DL, LOOKBACK)
Xts_seq, yts_seq, meta_ts = build_sequences_from_df(test_df_scaled,  FEATURES_DL, LOOKBACK)

print("\nSequence shapes:")
print("Xtr_seq:", Xtr_seq.shape, " ytr_seq:", ytr_seq.shape)
print("Xts_seq:", Xts_seq.shape, " yts_seq:", yts_seq.shape)

# Torch tensors for CNN (Conv1d expects: (batch, channels, seq_len))
Xtr_cnn = torch.tensor(np.transpose(Xtr_seq, (0, 2, 1)), dtype=torch.float32)
ytr_cnn = torch.tensor(ytr_seq, dtype=torch.float32).view(-1, 1)
Xts_cnn = torch.tensor(np.transpose(Xts_seq, (0, 2, 1)), dtype=torch.float32)
yts_cnn_np = np.asarray(yts_seq).ravel()

cnn_loader = DataLoader(TensorDataset(Xtr_cnn, ytr_cnn), batch_size=2048, shuffle=True)

class CNN1D(nn.Module):
    def __init__(self, n_features, lookback):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),   # -> (B, 64, 1)
            nn.Flatten(),             # -> (B, 64)
            nn.Dropout(0.15),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

print("\n====================")
print(f"Training: 1D-CNN (LOOKBACK={LOOKBACK})")
print("====================")
cnn = CNN1D(n_features=len(FEATURES_DL), lookback=LOOKBACK).to(device)
cnn_mse, cnn_r2, cnn_ic, cnn_pred, cnn_hist = early_stop_train(
    cnn, cnn_loader, Xts_cnn, yts_cnn_np, lr=1e-3, weight_decay=1e-4
)

print("\n✅ Best 1D-CNN")
print("Test MSE:", cnn_mse)
print("Test R2 :", cnn_r2)
print("Test IC :", cnn_ic)

# Summary table
dl_compare = pd.DataFrame([
    {"model": "TABULAR_MLP", "OOS_R2": mlp_r2, "MSE": mlp_mse, "IC": mlp_ic},
    {"model": "SEQ_1D_CNN",  "OOS_R2": cnn_r2, "MSE": cnn_mse, "IC": cnn_ic},
]).sort_values("IC", ascending=False)

display(dl_compare)


Device: cpu

Training: TABULAR MLP
Epoch 01 | train_MSE=0.011784 | test_MSE=0.008449 | test_R2=-0.0302 | test_IC=0.0385
Epoch 02 | train_MSE=0.008068 | test_MSE=0.008318 | test_R2=-0.0142 | test_IC=0.0328
Epoch 03 | train_MSE=0.007664 | test_MSE=0.008287 | test_R2=-0.0105 | test_IC=0.0406
Epoch 04 | train_MSE=0.007460 | test_MSE=0.008328 | test_R2=-0.0154 | test_IC=0.0333
Epoch 05 | train_MSE=0.007313 | test_MSE=0.008383 | test_R2=-0.0222 | test_IC=0.0304
Epoch 06 | train_MSE=0.007187 | test_MSE=0.008431 | test_R2=-0.0279 | test_IC=0.0312
Epoch 07 | train_MSE=0.007080 | test_MSE=0.008470 | test_R2=-0.0327 | test_IC=0.0286
Epoch 08 | train_MSE=0.006987 | test_MSE=0.008434 | test_R2=-0.0283 | test_IC=0.0488
⏹️ Early stopping triggered.

✅ Best TABULAR MLP
Test MSE: 0.008287260647328574
Test R2 : -0.010476604397735345
Test IC : 0.04062186418112282

Sequence shapes:
Xtr_seq: (168700, 60, 18)  ytr_seq: (168700,)
Xts_seq: (68900, 60, 18)  yts_seq: (68900,)

Training: 1D-CNN (LOOKBACK=60)
Epo

,model,OOS_R2,MSE,IC
0,TABULAR_MLP,-0.010477,0.008287,0.040622
1,SEQ_1D_CNN,-0.064601,0.008535,-0.007142


In [20]:
# %% Cell 7A — Build raw daily return sequences for CNN (Option A)

import numpy as np
import pandas as pd

LOOKBACK = 60  # try 60 first; later you can compare 120

# 1) daily simple returns from prices
ret1d = prices.pct_change().replace([np.inf, -np.inf], np.nan)

# 2) Build a (Date, Ticker) panel for returns so we can align to train/test rows
ret_panel = (
    ret1d.stack()
        .rename("ret1d")
        .reset_index()
        .rename(columns={"level_0": "Date", "level_1": "Ticker"})
)

# Ensure Date dtype matches your train/test Date dtype
ret_panel["Date"] = pd.to_datetime(ret_panel["Date"])
train_dl["Date"] = pd.to_datetime(train_dl["Date"])
test_dl["Date"]  = pd.to_datetime(test_dl["Date"])

# 3) Merge y with daily returns (y comes from your target panel)
train_ret = pd.merge(train_dl[["Date","Ticker","y"]], ret_panel, on=["Date","Ticker"], how="left")
test_ret  = pd.merge(test_dl[["Date","Ticker","y"]],  ret_panel, on=["Date","Ticker"], how="left")

train_ret = train_ret.sort_values(["Ticker","Date"]).reset_index(drop=True)
test_ret  = test_ret.sort_values(["Ticker","Date"]).reset_index(drop=True)

def build_return_sequences(df, lookback):
    """
    For each ticker: build sequences of last `lookback` daily returns ending at date t,
    predict y at date t.

    Returns:
      X: (N, lookback) float32
      y: (N,) float32
      meta: DataFrame(Date, Ticker) for each sample
    """
    X_list, y_list, meta_list = [], [], []

    for tick, g in df.groupby("Ticker", sort=False):
        g = g.sort_values("Date").reset_index(drop=True)

        r = g["ret1d"].values.astype(np.float64)
        y = g["y"].values.astype(np.float64)
        d = g["Date"].values

        for t in range(lookback - 1, len(g)):
            x_win = r[t - lookback + 1 : t + 1]
            y_t   = y[t]

            # require full window and target
            if np.isnan(y_t) or np.any(np.isnan(x_win)) or np.any(np.isinf(x_win)):
                continue

            X_list.append(x_win)
            y_list.append(y_t)
            meta_list.append((d[t], tick))

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.float32)
    meta = pd.DataFrame(meta_list, columns=["Date","Ticker"])
    return X, y, meta

Xtr_r, ytr_r, meta_tr = build_return_sequences(train_ret, LOOKBACK)
Xts_r, yts_r, meta_ts = build_return_sequences(test_ret,  LOOKBACK)

print("✅ Raw return sequence shapes:")
print("Xtr_r:", Xtr_r.shape, "ytr_r:", ytr_r.shape)
print("Xts_r:", Xts_r.shape, "yts_r:", yts_r.shape)


✅ Raw return sequence shapes:
Xtr_r: (168700, 60) ytr_r: (168700,)
Xts_r: (68900, 60) yts_r: (68900,)


In [21]:
# %% Cell 7B — Train 1D-CNN on raw return sequences (Option A)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Standardize returns using TRAIN ONLY
mu = Xtr_r.mean()
sd = Xtr_r.std(ddof=0) + 1e-12
Xtr_rs = (Xtr_r - mu) / sd
Xts_rs = (Xts_r - mu) / sd

# Convert to CNN format: (B, C, T) with C=1
Xtr_t = torch.tensor(Xtr_rs[:, None, :], dtype=torch.float32)
ytr_t = torch.tensor(ytr_r, dtype=torch.float32).view(-1, 1)
Xts_t = torch.tensor(Xts_rs[:, None, :], dtype=torch.float32)

yts_np = yts_r.copy()

train_loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=2048, shuffle=True)

class ReturnCNN(nn.Module):
    def __init__(self, lookback):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.15),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        return self.net(x)

def eval_model(model):
    model.eval()
    with torch.no_grad():
        pred = model(Xts_t.to(device)).cpu().numpy().ravel()
    mse = mean_squared_error(yts_np, pred)
    r2  = r2_score(yts_np, pred)
    ic  = spearmanr(pred, yts_np, nan_policy="omit").correlation
    return mse, r2, ic, pred

# Training with early stopping
EPOCHS = 30
PATIENCE = 5
best_mse = np.inf
best_state = None
pat = 0

model = ReturnCNN(LOOKBACK).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

hist = []
print("\n===============================")
print("Training: 1D-CNN on RAW RETURNS")
print("===============================")

for ep in range(1, EPOCHS + 1):
    model.train()
    total = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = loss_fn(out, yb)
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item() * len(xb)

    train_mse = total / len(train_loader.dataset)
    test_mse, test_r2, test_ic, _ = eval_model(model)

    hist.append({"epoch": ep, "train_mse": train_mse, "test_mse": test_mse, "test_r2": test_r2, "test_ic": test_ic})
    print(f"Epoch {ep:02d} | train_MSE={train_mse:.6f} | test_MSE={test_mse:.6f} | test_R2={test_r2:.4f} | test_IC={test_ic:.4f}")

    if test_mse < best_mse - 1e-6:
        best_mse = test_mse
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        pat = 0
    else:
        pat += 1
        if pat >= PATIENCE:
            print("⏹️ Early stopping triggered.")
            break

# Load best
if best_state is not None:
    model.load_state_dict(best_state)

best_mse, best_r2, best_ic, best_pred = eval_model(model)

print("\n✅ Best RAW-RET CNN results")
print("Test MSE:", best_mse)
print("Test R2 :", best_r2)
print("Test IC :", best_ic)

cnn_ret_hist = pd.DataFrame(hist)
display(cnn_ret_hist.tail(10))


Device: cpu

Training: 1D-CNN on RAW RETURNS
Epoch 01 | train_MSE=0.008450 | test_MSE=0.007900 | test_R2=0.0146 | test_IC=0.0766
Epoch 02 | train_MSE=0.007927 | test_MSE=0.007920 | test_R2=0.0121 | test_IC=0.0709
Epoch 03 | train_MSE=0.007841 | test_MSE=0.007936 | test_R2=0.0101 | test_IC=0.0633
Epoch 04 | train_MSE=0.007770 | test_MSE=0.007982 | test_R2=0.0044 | test_IC=0.0585
Epoch 05 | train_MSE=0.007703 | test_MSE=0.007996 | test_R2=0.0026 | test_IC=0.0427
Epoch 06 | train_MSE=0.007619 | test_MSE=0.008088 | test_R2=-0.0088 | test_IC=0.0371
⏹️ Early stopping triggered.

✅ Best RAW-RET CNN results
Test MSE: 0.007900282
Test R2 : 0.014562487602233887
Test IC : 0.07663776407333188


,epoch,train_mse,test_mse,test_r2,test_ic
0,1,0.008450,0.007900,0.014562,0.076638
1,2,0.007927,0.007920,0.012054,0.070946
2,3,0.007841,0.007936,0.010128,0.063309
3,4,0.007770,0.007982,0.004366,0.058451
4,5,0.007703,0.007996,0.002566,0.042743
5,6,0.007619,0.008088,-0.008823,0.037123


In [22]:
# %% Cell 7C — IC by date + IC-IR (RAW-RET CNN)

test_pred_df = meta_ts.copy()
test_pred_df["y"] = yts_np
test_pred_df["pred"] = best_pred
test_pred_df["Date"] = pd.to_datetime(test_pred_df["Date"])

ic_by_date = (
    test_pred_df.groupby("Date")
                .apply(lambda g: g["pred"].corr(g["y"], method="spearman"))
                .dropna()
)

ic_mean = float(ic_by_date.mean())
ic_std  = float(ic_by_date.std(ddof=0))
ic_ir   = float(ic_mean / (ic_std + 1e-12))

print("✅ RAW-RET CNN IC mean:", ic_mean)
print("✅ RAW-RET CNN IC IR  :", ic_ir)
print("N dates used         :", len(ic_by_date))

display(ic_by_date.describe())


✅ RAW-RET CNN IC mean: 0.05300658123486017
✅ RAW-RET CNN IC IR  : 0.23134838358551907
N dates used         : 689


count    689.000000
mean       0.053007
std        0.229287
min       -0.381638
25%       -0.105035
50%        0.028983
75%        0.184926
max        0.721908
dtype: float64